In [70]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader


In [71]:
t = torch.tensor([[1,2],[3,4]])
print(t)
print(t.shape)
print(t.dtype)

tensor([[1, 2],
        [3, 4]])
torch.Size([2, 2])
torch.int64


In [72]:
device = torch.device("mps")
print(t.device)

t_gpu = t.to(device)
print(t_gpu.device)

cpu
mps:0


In [73]:
fake = torch.rand(1,1,64,64)

conv1 = nn.Conv2d(in_channels = 1, out_channels = 16, kernel_size = 3)
out = conv1(fake)
print(out.shape)

torch.Size([1, 16, 62, 62])


In [74]:
relu = nn.ReLU()
pool = nn.MaxPool2d(kernel_size = 2)

out = pool(relu(conv1(fake)))
print(out.shape)
print(relu(conv1(fake)).min())


torch.Size([1, 16, 31, 31])
tensor(0., grad_fn=<MinBackward1>)


In [75]:
conv2 = nn.Conv2d(in_channels = 16, out_channels = 32, kernel_size = 3)
out = pool(relu(conv2(out)))
print(out.shape)

torch.Size([1, 32, 14, 14])


In [76]:
flatten = nn.Flatten()
fc = nn.Linear(in_features = 32 * 14 * 14, out_features = 9)
flat = flatten(out)
scores = fc(flat)
print(flat.shape)
print(scores.shape)

torch.Size([1, 6272])
torch.Size([1, 9])


In [77]:
class WaferCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3)
        self.conv2 = nn.Conv2d(16, 32, 3)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(32 * 14 * 14, 9)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.flatten(x)
        x = self.fc(x)
        return x
        
model = WaferCNN()
print(model(fake).shape)

torch.Size([1, 9])


In [78]:
X = np.load("/Users/a./Wafer_project/wafer-defects/data/X.npy")
y = np.load("/Users/a./Wafer_project/wafer-defects/data/y.npy")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=67
)
print(X_train.shape, X_test.shape)

(28415, 64, 64) (7104, 64, 64)


In [79]:
classes = np.unique(y).tolist()
class_to_idx = {name: i for i, name in enumerate(classes)}
print(class_to_idx)

{'Center': 0, 'Donut': 1, 'Edge-Loc': 2, 'Edge-Ring': 3, 'Loc': 4, 'Near-full': 5, 'Random': 6, 'Scratch': 7, 'none': 8}


In [80]:
y_train_idx = np.array([class_to_idx[c]for c in y_train])
y_test_idx = np.array([class_to_idx[c] for c in y_test])

print(y_train[:5])
print(y_train_idx[:5])

['Random' 'none' 'none' 'Scratch' 'none']
[6 8 8 7 8]


In [81]:
X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)

y_train_t = torch.tensor(y_train_idx, dtype=torch.long)
y_test_t = torch.tensor(y_test_idx, dtype=torch.long)

print(X_train_t.shape, y_train_t.shape)

torch.Size([28415, 1, 64, 64]) torch.Size([28415])


In [82]:
number = y_train_t[0].item()
print(classes[number])

Random


In [83]:
train_ds = TensorDataset(X_train_t,y_train_t) 
test_ds = TensorDataset(X_test_t,y_test_t)
train_loader = DataLoader(train_ds,batch_size = 64, shuffle = True)
test_loader = DataLoader(test_ds,batch_size = 64, shuffle = False)

xb, yb = next(iter(train_loader))
print(xb.shape, yb.shape)

torch.Size([64, 1, 64, 64]) torch.Size([64])


In [85]:
model = WaferCNN()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# before learning
logits = model(xb)
loss = loss_fn(logits, yb)
print("before:", loss)

# one learning step
optimizer.zero_grad()
loss.backward()
optimizer.step()

# after learning
logits = model(xb)
loss = loss_fn(logits, yb)
print("after:", loss)

before: tensor(2.1914, grad_fn=<NllLossBackward0>)
after: tensor(1.8890, grad_fn=<NllLossBackward0>)


In [86]:
-np.log(1/9)

np.float64(2.1972245773362196)